# Coding-agent routing: baseline lab

Start with contracts and synthetic probes, then review real examples before making benchmark claims. This notebook defaults to the offline rules baseline. **Clef weights stay on Azure: use `clef-endpoint` from this local notebook, or `clef` when the notebook itself runs on Azure.** No fine-tuning or downstream code execution happens here.

Setup: follow `README.md` and `docs/deployment.md`, select the project's Python 3.11 `.venv` kernel, and run the cells in order. Run outputs are saved under `outputs/`. Do not check private inputs, executed sensitive cells, or credentials into source control.

In [ ]:
from pathlib import Path
import json
import sys
import time
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'contracts' / 'agents.json').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the project directory or notebooks directory')
sys.path.insert(0, str(ROOT))
from routing_lab.data import Record, load_contracts, load_records, write_records, split_reviewed, assert_disjoint
from routing_lab.backends import RulesBackend, ConstantBackend, ClefBackend, TfidfBackend, AzureLLMBackend
from routing_lab.endpoint import ClefEndpointBackend
from routing_lab.evaluation import run_experiment
from routing_lab.outcomes import compare_outcomes
contracts = load_contracts(ROOT / 'contracts' / 'agents.json')
records = load_records(ROOT / 'data' / 'probes.jsonl')
print(f'{len(records)} unreviewed synthetic examples; Python {sys.version.split()[0]}')

In [ ]:
for route, contract in contracts.items():
    print(f'{route}: {contract.description}')
    print(f'  tools={contract.tools}; can_write={contract.can_write}')
print('\nThese are capability descriptions, not runtime permission grants.')
print(records[0].model_dump())

## Choose a backend

Leave `BACKEND = 'rules'` for an offline smoke test. For a local notebook calling Azure, install `uv sync --frozen --extra endpoint`, run `az login`, then select `clef-endpoint` and set `ALLOW_CLEF_ENDPOINT = True`. The backend cell reads the provisioned scoring URI from `azure/online/connection.json`. No local model installation is needed. Endpoint latency includes authentication and the network. See `docs/deployment.md` for deployment and verification status.

Alternatively, on an Azure GPU notebook install the `clef` dependency group (`uv pip install --group clef`), select `clef` and confirm `RUNNING_ON_AZURE`. Clef-Flash (9B) is the first base test; set `variant='clef'` for the 27B model, which needs an 80 GB-class GPU.

Use `context_mode='query'` versus `'context'`, and `reverse=True`, in separate runs to probe context and option-order sensitivity. Check recorded token truncation before interpreting errors. Setting `RUNNING_ON_AZURE` is an explicit user confirmation, not cloud-environment detection.

In [ ]:
BACKEND = 'clef-endpoint'
ALLOW_CLEF_ENDPOINT = True
RUNNING_ON_AZURE = False
DEVICE = 'cuda'
start = time.perf_counter()
if BACKEND == 'rules':
    backend = RulesBackend()
elif BACKEND == 'clef-endpoint':
    if not ALLOW_CLEF_ENDPOINT:
        raise RuntimeError('Confirm approved data and set ALLOW_CLEF_ENDPOINT before remote calls')
    connection = json.loads((ROOT / 'azure' / 'online' / 'connection.json').read_text())
    backend = ClefEndpointBackend(contracts, url=connection['scoring_uri'], context_mode='context')
elif BACKEND == 'clef':
    if not RUNNING_ON_AZURE:
        raise RuntimeError('Provision Azure and confirm RUNNING_ON_AZURE before loading weights')
    backend = ClefBackend(contracts, device=DEVICE, context_mode='context', variant='flash')
else:
    raise ValueError('Choose rules, clef-endpoint or clef; see the reviewed comparison section for other baselines')
initialization_seconds = time.perf_counter() - start
backend.metadata()

## One example: exactly what goes into Clef and comes back

`backend.metadata()` describes the backend configuration; it does **not** call the model. The next cells make **one live request**, separate from the 60-example probe below. Run the setup and backend-selection cells first with `BACKEND = 'clef-endpoint'` and `ALLOW_CLEF_ENDPOINT = True`.

We use the first synthetic example: fix an empty-list division-by-zero crash and add a regression test. The request has two fields:

- **`state`**: the user's request plus available code/context, built with the same helper as `backend.predict()`.
- **`questions.route`**: a `choice` question, shared routing instructions, and six `criteria` mapping route names to agent descriptions. These are the choices to score, not six separate user requests.

On Azure, the scorer validates this JSON and calls `agent.system_one(request.state, request.questions.model_dump())`. The expected route, dataset ID, and review labels are **not** sent. Authentication stays in an HTTP header, which is never displayed here.

In [ ]:
from routing_lab.backends import state

example = records[0]
example_request = None
example_response = None
if isinstance(backend, ClefEndpointBackend):
    example_request = {
        'state': state(example, backend.context_mode),
        'questions': backend.question,
    }
    print(f'Example: {example.id} (synthetic; proposed labels are unreviewed)')
    print('REQUEST BODY sent to the Azure endpoint:')
    print(json.dumps(example_request, indent=2))
else:
    print('Live walkthrough skipped: select clef-endpoint in the backend cell first.')

In [ ]:
example_response = None
if isinstance(backend, ClefEndpointBackend) and example_request is not None:
    if not ALLOW_CLEF_ENDPOINT:
        raise RuntimeError('Confirm approved data and set ALLOW_CLEF_ENDPOINT before remote calls')
    import requests
    from routing_lab.endpoint import TOKEN_SCOPE

    example_started = time.perf_counter()
    with requests.post(
        backend.url,
        json=example_request,
        headers={
            'Authorization': f'Bearer {backend.credential.get_token(TOKEN_SCOPE).token}',
            'Content-Type': 'application/json',
        },
        timeout=(10, 190), allow_redirects=False,
    ) as response:
        if response.status_code != 200:
            raise RuntimeError(f'Clef endpoint returned HTTP {response.status_code}; check deployment health and scoring permissions')
        example_payload = response.json()
    if not isinstance(example_payload, dict) or example_payload.get('experiment_model') != backend.expected_model:
        raise ValueError('Endpoint model provenance differs from the pinned Clef release')
    example_response = example_payload
    print(f'Client elapsed time: {time.perf_counter() - example_started:.3f}s (includes authentication and network)')
    print('ACTUAL RESPONSE BODY from the Azure endpoint:')
    print(json.dumps(example_response, indent=2))
else:
    print('No request sent: run the request-preview cell with the endpoint backend selected.')

### Read the response

- **`answers.route.choice`** is the selected owning agent; **`probabilities`** contains scores for all six routes. These probabilities are not validated confidence or a guarantee of correctness.
- Native **`confidence`** is the selected option's probability, displayed unchanged. Do not interpret it as measured task success or permission to execute tools.
- **`usage`** reports input tokens and how many state tokens were dropped to fit `max_length`. A choice response is not a generated code solution or a reasoning explanation.
- **`experiment_model`** is provenance added by our Azure scorer, not a native model answer. The client checks it against the pinned model.

The comparison below uses the proposed label only *after* inference. A mismatch is useful evidence to review; it is not proof that the model or the unreviewed label is correct. Edit the example/input and rerun the two code cells above to explore another approved request.

In [ ]:
if example_response is not None:
    example_answer = example_response['answers']['route']
    print('Selected agent:', example_answer['choice'])
    print('Proposed label (NOT sent to Clef):', example.expected_route)
    print('Matches proposed label:', example_answer['choice'] == example.expected_route)
    print('\nOption probabilities, highest first:')
    for option, probability in sorted(example_answer['probabilities'].items(), key=lambda item: item[1], reverse=True):
        print(f'  {option:20s} {probability:.4f}')
    print('\nUsage:', json.dumps(example_response['usage'], indent=2))
else:
    print('No live response yet; run the request and response cells above.')

In [ ]:
run_dir = ROOT / 'outputs' / f'notebook-{BACKEND}-{uuid4().hex[:10]}'
summary = run_experiment(backend, records, run_dir, probe=True,
                         initialization_seconds=initialization_seconds,
                         source_paths={'dataset': ROOT / 'data' / 'probes.jsonl',
                                       'contracts': ROOT / 'contracts' / 'agents.json'})
print(run_dir)
print(json.dumps({k: summary[k] for k in ['mode', 'count', 'strict_accuracy', 'macro_f1', 'latency_ms', 'truncation_rate']}, indent=2))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(summary['confusion_matrix'], cmap='Blues')
ax.set_xticks(range(6), summary['route_order'], rotation=45, ha='right')
ax.set_yticks(range(6), summary['route_order'])
ax.set(xlabel='Predicted route', ylabel='Proposed label', title='Synthetic probe only, not a reviewed benchmark')
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()
if 'probability_metrics' in summary:
    print(json.dumps(summary['probability_metrics'], indent=2))
else:
    print('No probability metrics: rules do not provide calibrated probabilities.')

In [ ]:
predictions = [json.loads(line) for line in (run_dir / 'predictions.jsonl').read_text().splitlines()]
errors = [p for p in predictions if p['decision']['route'] not in p['acceptable_routes']]
by_id = {r.id: r for r in records}
for p in errors[:10]:
    print(p['id'], 'proposed=', p['expected_route'], 'predicted=', p['decision']['route'])
    print(by_id[p['id']].query)
print('Review labeling ambiguity, missing context, truncation and model error separately.')

## Try your own request

This is an unlabeled interactive prediction. The temporary record's fallback label is not used by the backend and is not scored. Only enter approved data; local notebook outputs can retain text.

In [ ]:
custom = Record(id='interactive', group_id='interactive', repository='interactive',
    query='Explain why this function returns None; do not change the code.',
    context='def lookup(items, key):\n    for item in items:\n        if item.key == key:\n            return item',
    expected_route='fallback', acceptable_routes=['fallback'], tags=['interactive_unlabeled'],
    provenance='interactive:user', review_status='unreviewed', reviewer='')
backend.predict(custom).model_dump()

## Prompt tuning before fine-tuning: richer criteria

Hypothesis: explicit route scope, inclusions, and exclusions improve routing. This is textual prompt engineering, not learned soft-prompt tuning or weight fine-tuning. We compare the original descriptions with versioned `contracts/criteria-expanded-v1.json`, leaving the model revision, user inputs, shared instructions, option order, and six route names unchanged. No example answers are inserted into the criteria.

The same 60 previously inspected, unreviewed synthetic probes are a **development set**, not a held-out benchmark. Report all corrections and regressions, not just the motivating example. Longer wording can hurt as well as help; examine both state truncation and question truncation. Since wording and length change together, a result cannot establish length alone as the cause.

The saved result can be inspected without another Azure call. To rerun, restart the kernel after library changes, run setup and backend selection, then enable the explicit switch below. A fresh comparison makes 120 remote requests (60 per arm) and creates a new output folder; it does not overwrite old runs or replace the default criteria.

Recorded run `prompt-criteria-1702b7ca67` (2026-09-30): both arms scored **30/60 (50%)**. Macro-F1 changed from **0.4601 to 0.4723**. Expanded criteria corrected 3 examples and regressed 3; 9 predictions changed in total. The motivating `dbg-01` changed from `code_generation` to `debugging`, but total agreement did not increase. Neither arm reported state or question truncation. This is a mixed result, not proof that more text solves routing.

See [experiment results and limitations](../docs/data-and-training.md#routing-criteria-prompt-tuning-experiment) and the [executed comparison notebook](../outputs/notebook-prompt-tuning-validated.ipynb). Reported token counts were identical; treat them as native usage telemetry, not an independent count of all criterion text.

In [ ]:
from routing_lab.prompt_experiment import expanded_contracts, run_prompt_comparison

criteria_path = ROOT / 'contracts' / 'criteria-expanded-v1.json'
prompt_baseline = load_contracts(ROOT / 'contracts' / 'agents.json')
prompt_expanded = expanded_contracts(prompt_baseline, criteria_path)
for route in prompt_baseline:
    print(f'\n{route}\nBASELINE: {prompt_baseline[route].description}\nEXPANDED: {prompt_expanded[route].description}')

In [ ]:
RUN_PROMPT_COMPARISON = False
PROMPT_RESULT_PATH = ROOT / 'outputs' / 'prompt-criteria-1702b7ca67' / 'comparison.json'
if RUN_PROMPT_COMPARISON:
    if BACKEND != 'clef-endpoint' or not ALLOW_CLEF_ENDPOINT:
        raise RuntimeError('Select clef-endpoint and approve remote data before running prompt experiments')
    prompt_output = ROOT / 'outputs' / f'prompt-criteria-{uuid4().hex[:10]}'
    prompt_connection = json.loads((ROOT / 'azure' / 'online' / 'connection.json').read_text())
    prompt_report = run_prompt_comparison(
        contracts_path=ROOT / 'contracts' / 'agents.json', criteria_path=criteria_path,
        data_path=ROOT / 'data' / 'probes.jsonl', url=prompt_connection['scoring_uri'],
        output=prompt_output, allow_remote=ALLOW_CLEF_ENDPOINT,
    )
    PROMPT_RESULT_PATH = prompt_output / 'comparison.json'
    print('Saved new comparison:', PROMPT_RESULT_PATH)
else:
    print('No remote calls. Reading the saved comparison below:', PROMPT_RESULT_PATH)

In [ ]:
prompt_report = None
if not PROMPT_RESULT_PATH.exists():
    print('No saved comparison at this path. Enable the experiment or select an existing comparison.json.')
else:
    prompt_report = json.loads(PROMPT_RESULT_PATH.read_text(encoding='utf-8'))
    if prompt_report['status'] != 'complete':
        raise RuntimeError('Comparison is incomplete; inspect the arm manifests before interpreting results')
    print(prompt_report['mode'])
    for name, result in prompt_report['results'].items():
        print(f'\n{name}: {result["strict_accuracy"]:.1%} proposed-label agreement, macro-F1 {result["macro_f1"]:.4f}')
        print(f'  Average input tokens: {result["input_tokens"] / result["count"]:.1f}; client p50: {result["latency_ms"]["p50"] / 1000:.3f}s')
        print('  State truncation:', result['truncation_rate'])
        print('  Recall by proposed route:', {route: round(values['recall'], 3) for route, values in result['per_route'].items()})
    print('\nExpanded minus baseline:', prompt_report['delta_expanded_minus_baseline'])
    print('Paired changes:', {key: prompt_report['paired'][key] for key in ['corrected', 'regressed', 'changed']})
    print('\nMotivating example and all changed predictions:')
    for change in prompt_report['paired']['records']:
        if change['id'] == 'dbg-01' or change['changed']:
            print(change)
    print('\nDo not select a production prompt or start fine-tuning on this score alone. Review labels, freeze the candidate, then evaluate held-out tasks and downstream outcomes.')

## Build and review the evaluation set

Collect authorized real requests with decision-time context. Keep parent issue IDs and repository identifiers. Have reviewers edit `expected_route`, `acceptable_routes`, `review_status` and `reviewer` in a separate JSONL copy; do not bulk mark generated examples reviewed. Record disagreements and adjudication in your review system. Keep all paraphrases together. The schema validates a reviewer name, not the truth of a human review.

The next cell is disabled until a reviewed file exists and you explicitly opt in. Inspect route counts after splitting. Add more independent repositories if any split lacks classes. Splitting catches exact normalized duplicates, not semantic near-duplicates. Keep benchmark test tasks out of training sources. `dev` is for prompt/threshold selection; reserve a separate calibration subset of dev if fitting probabilities. Freeze test before comparison.

In [ ]:
CREATE_REVIEWED_SPLITS = False
reviewed_path = ROOT / 'data' / 'reviewed' / 'routing.jsonl'
split_dir = ROOT / 'data' / 'splits'
if CREATE_REVIEWED_SPLITS:
    from collections import Counter
    reviewed = load_records(reviewed_path, reviewed_only=True)
    splits = split_reviewed(reviewed)
    split_dir.mkdir(parents=True, exist_ok=False)
    for name, subset in splits.items():
        write_records(split_dir / f'{name}.jsonl', subset)
        print(name, dict(Counter(r.expected_route for r in subset)))
else:
    print('Pending: human-reviewed data. No benchmark or training data fabricated.')

## Compare baselines on reviewed data

Use dev while iterating. Change to test only after freezing choices. Rules and constant fallback need no fit; majority and TF-IDF fit on reviewed train. Run the CLI for majority. Azure OpenAI is optional and separately gated below; configure the three environment variables documented in the Azure guide and authorize the executing identity. It sends input text to that deployment. LLM self-reported confidence is intentionally not collected.

In [ ]:
RUN_REVIEWED_COMPARISON = False
ALLOW_AZURE_OPENAI = False
if RUN_REVIEWED_COMPARISON:
    train = load_records(split_dir / 'train.jsonl', reviewed_only=True)
    evaluation = load_records(split_dir / 'dev.jsonl', reviewed_only=True)
    assert_disjoint(train, evaluation)
    candidates = {'rules': RulesBackend(), 'constant': ConstantBackend(), 'tfidf': TfidfBackend(train)}
    if BACKEND in {'clef', 'clef-endpoint'}:
        candidates[BACKEND] = backend
    if ALLOW_AZURE_OPENAI:
        candidates['azure'] = AzureLLMBackend(contracts)
    for name, candidate in candidates.items():
        report = run_experiment(candidate, evaluation,
            ROOT / 'outputs' / f'reviewed-{name}-{uuid4().hex[:10]}',
            source_paths={'training': split_dir / 'train.jsonl', 'dataset': split_dir / 'dev.jsonl',
                          'contracts': ROOT / 'contracts' / 'agents.json'})
        print(name, report['macro_f1'], report['acceptable_accuracy'], report['latency_ms'])
else:
    print('Pending: frozen reviewed splits.')

## Downstream outcomes and the fine-tuning decision

Run real agents separately in isolated Azure environments. This project does not execute issue code or implement the five solving agents. Compare always-general versus routed policies on the same tasks using a common grader and identical budgets. Record routing, context construction, fallback, all retries and hosting costs in total cost. Import graded results with the schema in `docs/data-and-training.md`.

Do not decide to fine-tune from synthetic probe accuracy. Require reviewed error analysis, a frozen test set, meaningful coverage, and downstream evidence. Confidence intervals from small or correlated task samples are descriptive, not production certification.

In [ ]:
COMPARE_REAL_OUTCOMES = False
if COMPARE_REAL_OUTCOMES:
    outcomes = compare_outcomes(ROOT / 'outputs' / 'downstream.jsonl', 'always_general', 'routed')
    print(json.dumps(outcomes, indent=2))
else:
    print('Pending: real, independently graded downstream runs. Fine-tuning remains a manual decision.')